In [1]:
# Import os to read environment variables like OPENAI_API_KEY
import os
# Import sys so we can target the active Jupyter notebook's Python interpreter
import sys

# Install openai and python-dotenv into this exact Anaconda kernel if not already present
!{sys.executable} -m pip install -q openai python-dotenv

# Import load_dotenv to load variables from your local .env file into the environment
from dotenv import load_dotenv
# Import OpenAI to initialize the official OpenAI client
from openai import OpenAI

# Load key-value pairs from .env without printing secrets to the notebook
load_dotenv()

# Stop early with a clear message if OPENAI_API_KEY is missing or empty
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env or environment"

# Confirm setup succeeded without displaying the secret key
print("Step 1 complete: Anaconda environment and OPENAI_API_KEY are ready.")

Step 1 complete: Anaconda environment and OPENAI_API_KEY are ready.


In [2]:
# Import time to add short delays between calls and back off on rate limits
import time
# Import OpenAI exception classes to catch rate limits and transient API errors
from openai import APIError, RateLimitError

# Create one shared OpenAI client that automatically reads OPENAI_API_KEY from the environment
client = OpenAI()

# Define a reusable helper function to send a prompt and return the model's text response
def call_openai(
    prompt: str,
    model: str = "gpt-5-mini",
    temperature: float | None = None,
    max_retries: int = 3,
    sleep_seconds: float = 1.0,
) -> str:
    # Retry up to max_retries times if the API hits temporary rate limits or server errors
    for attempt in range(max_retries):
        try:
            # Start building the request parameters for the Responses API
            request_params = {
                "model": model,
                "input": prompt,
            }
            # Include temperature only when explicitly specified so reasoning models do not reject it
            if temperature is not None:
                request_params["temperature"] = temperature

            # Send the prompt to OpenAI and wait for the response object
            response = client.responses.create(**request_params)
            # Pause briefly so repeated loops stay gentle on rate limits
            time.sleep(sleep_seconds)
            # Return the extracted text output from the response
            return response.output_text

        # Handle HTTP 429 rate-limit responses by waiting longer before retrying
        except RateLimitError:
            # Wait with linear backoff if retries remain
            if attempt < max_retries - 1:
                time.sleep(sleep_seconds * (attempt + 2))
            else:
                # Re-raise once retries are used up so you can see the error
                raise

        # Handle transient API errors by waiting briefly and retrying
        except APIError:
            # Pause before retrying if attempts remain
            if attempt < max_retries - 1:
                time.sleep(sleep_seconds)
            else:
                raise

In [3]:
# Send a short test prompt to verify API connectivity and helper behavior
smoke_response = call_openai("Respond with the single word: READY")
# Print the returned text to confirm everything works as expected
print("Smoke test result:", smoke_response)

Smoke test result: READY


In [4]:
# Define the baseline zero-shot v1 prompt for classifying customer sentiment
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

# Run the v1 sentiment prompt once using our reusable OpenAI helper
sentiment_v1_result = call_openai(sentiment_prompt_v1)
# Print a clear header so single-run outputs stay easy to scan in the notebook
print("Sentiment Analysis (v1) Result:")
# Display the model's raw response to inspect formatting and tone
print(sentiment_v1_result)

Sentiment Analysis (v1) Result:
Sentiment: Positive.

Reason: Expresses strong satisfaction and praise ("I love this product", "exactly what I needed") — a clear endorsement/no complaint.


In [5]:
# Define the baseline zero-shot v1 prompt for generating product copy with price info
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

# Run the v1 product description prompt once to see initial structure and length
product_v1_result = call_openai(product_prompt_v1)
# Print a clear header for the product description output
print("Product Description (v1) Result:")
# Display the generated description to check what features or phrasing appear
print(product_v1_result)

Product Description (v1) Result:
Sleek, responsive, and built for everyday use — this wireless mouse delivers precise control and all-day comfort at an unbeatable price.

Key features
- Smooth tracking with adjustable DPI (800 / 1200 / 1600) for browsing, spreadsheets, and light gaming
- Dual-mode wireless: 2.4 GHz USB receiver for plug-and-play reliability, plus Bluetooth for laptop and tablet pairing
- Ergonomic, contoured shape with soft-touch finish for comfortable use during long sessions
- Whisper-quiet clicks and a tactile scroll wheel for distraction-free work in shared spaces
- Long battery life with auto-sleep power saving; easy battery access for quick replacement
- Broad compatibility: Windows, macOS, Chrome OS, and Linux

Technical specs
- Connection: 2.4 GHz USB dongle + Bluetooth 5.0
- DPI: 800 / 1200 / 1600 (switchable)
- Buttons: Left / Right / Scroll wheel / DPI button
- Power: Single AA battery (included) — months of typical use
- Dimensions: Compact, travel-friendly

In [6]:
# Define the baseline zero-shot v1 prompt for extracting fields from customer feedback
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

# Run the v1 extraction prompt once to see what format and fields it picks by default
extraction_v1_result = call_openai(extraction_prompt_v1)
# Print a clear header for the data extraction output
print("Data Extraction (v1) Result:")
# Display the extracted output to check structure, keys, and completeness
print(extraction_v1_result)

Data Extraction (v1) Result:
{
  "order_number": "12345",
  "order_date": "March 15",
  "delivery_speed": "fast",
  "packaging_condition": "damaged",
  "issue": "damaged packaging",
  "overall_sentiment": "mixed (positive about delivery speed, negative about packaging)",
  "customer_request": null,
  "original_feedback": "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
}


In [7]:
# Define a helper function to run the same prompt n times and collect all outputs
def run_multiple(
    prompt: str,
    runs: int = 5,
    model: str = "gpt-5-mini",
    temperature: float | None = None,
) -> list[str]:
    # Initialize a list to store every response in execution order
    results = []
    # Repeat the prompt call for the requested number of iterations
    for _ in range(runs):
        # Call our shared OpenAI helper and append the resulting text to results
        results.append(call_openai(prompt=prompt, model=model, temperature=temperature))
    # Return the full list of outputs for inspection and consistency analysis
    return results

# Run the v1 sentiment prompt 5 times to observe format and wording variation
sentiment_v1_5runs = run_multiple(sentiment_prompt_v1, runs=5)
# Print the number of distinct responses out of 5 runs
print(f"Sentiment v1 (5 runs) — unique outputs: {len(set(sentiment_v1_5runs))}/5")
# Print each individual run so you can compare format and phrasing side by side
for idx, output in enumerate(sentiment_v1_5runs, start=1):
    # Show the run number and the returned text
    print(f"[Run {idx}] {output}\n")

# Run the v1 product description prompt 5 times to check length and spec consistency
product_v1_5runs = run_multiple(product_prompt_v1, runs=5)
# Print how many unique responses came back across the 5 runs
print(f"Product v1 (5 runs) — unique outputs: {len(set(product_v1_5runs))}/5")
# Print a short preview of each run so the notebook output stays readable
for idx, output in enumerate(product_v1_5runs, start=1):
    # Show the first 180 characters of each generated description
    print(f"[Run {idx}] {output[:180].replace(chr(10), ' ')}...\n")

# Run the v1 data extraction prompt 5 times to check JSON and key stability
extraction_v1_5runs = run_multiple(extraction_prompt_v1, runs=5)
# Print how many unique outputs appeared across the 5 runs
print(f"Extraction v1 (5 runs) — unique outputs: {len(set(extraction_v1_5runs))}/5")
# Print each extraction result to compare schema keys and values across runs
for idx, output in enumerate(extraction_v1_5runs, start=1):
    # Display the run index and full extracted response
    print(f"[Run {idx}]\n{output}\n")

Sentiment v1 (5 runs) — unique outputs: 5/5
[Run 1] Sentiment: Positive.

Tone/intent: Enthusiastic praise / satisfied customer (expresses that the product meets their needs).

[Run 2] Positive — very satisfied/enthusiastic customer.

[Run 3] Label: Positive (strong)

Reason: The message expresses clear satisfaction and enthusiasm ("I love this product!" and "exactly what I needed").

[Run 4] Positive — clearly satisfied/enthusiastic about the product.

[Run 5] Sentiment: Positive (very positive / enthusiastic)

Category tags: Praise, Customer satisfaction, Product appreciation

Why: Expresses strong liking ("I love this product") and confirms it meets the customer's needs.

Suggested agent response: "Thanks so much — we're thrilled it’s exactly what you needed! If you have any feedback or questions, we’d love to hear from you."

Product v1 (5 runs) — unique outputs: 5/5
[Run 1] Wireless Comfort. Precise Control. Perfect Price — $29.99.  Take control of your workday and play sessions w

In [8]:
# Update call_openai default sleep slightly so 10-run batches complete faster while staying rate-limit safe
def run_multiple_fast(
    prompt: str,
    runs: int = 10,
    model: str = "gpt-5-mini",
    sleep_seconds: float = 0.3,
) -> list[str]:
    # Store every returned response string across all iterations
    results = []
    # Execute the prompt runs times in sequence
    for _ in range(runs):
        # Call our shared helper with a 0.3s pause between requests
        results.append(call_openai(prompt=prompt, model=model, sleep_seconds=sleep_seconds))
    # Return the complete list of responses
    return results

# Run the v1 sentiment prompt 10 times to measure format drift at larger sample size
sentiment_v1_10runs = run_multiple_fast(sentiment_prompt_v1, runs=10)
# Print the unique-output ratio out of 10 runs
print(f"Sentiment v1 (10 runs) — unique outputs: {len(set(sentiment_v1_10runs))}/10")
# Display each run's first line so you can compare labels and prefixes quickly
for idx, output in enumerate(sentiment_v1_10runs, start=1):
    # Extract the first non-empty line for a compact comparison
    first_line = output.strip().splitlines()[0]
    # Print the run number and opening line
    print(f"  [Run {idx:02d}] {first_line}")

# Run the v1 product description prompt 10 times to track invented specs/brands and structure shifts
product_v1_10runs = run_multiple_fast(product_prompt_v1, runs=10)
# Print how many unique descriptions were generated out of 10
print(f"\nProduct v1 (10 runs) — unique outputs: {len(set(product_v1_10runs))}/10")
# Print the opening headline and character length of each run
for idx, output in enumerate(product_v1_10runs, start=1):
    # Grab the first line to inspect headline and invented product names
    headline = output.strip().splitlines()[0]
    # Print run number, total length in characters, and headline
    print(f"  [Run {idx:02d}] ({len(output)} chars) {headline[:90]}")

# Run the v1 data extraction prompt 10 times to check JSON validity and key drift
extraction_v1_10runs = run_multiple_fast(extraction_prompt_v1, runs=10)
# Print how many unique extraction responses occurred out of 10
print(f"\nExtraction v1 (10 runs) — unique outputs: {len(set(extraction_v1_10runs))}/10")
# Import json to test whether each output parses as valid JSON and inspect top-level keys
import json
# Track how many of the 10 runs parse cleanly as JSON
valid_json_count = 0
# Inspect each run's JSON validity and top-level schema keys
for idx, output in enumerate(extraction_v1_10runs, start=1):
    try:
        # Attempt to parse the model's response as JSON
        parsed = json.loads(output)
        # Increment valid JSON counter when parsing succeeds
        valid_json_count += 1
        # Print the sorted top-level keys to expose schema drift across runs
        print(f"  [Run {idx:02d}] Valid JSON | keys={sorted(parsed.keys())}")
    except json.JSONDecodeError:
        # Flag runs that returned prose or markdown fences instead of raw JSON
        print(f"  [Run {idx:02d}] INVALID JSON | starts with: {output.strip()[:60]!r}")

# Summarize JSON parse rate across all 10 baseline extraction runs
print(f"Extraction v1 valid JSON rate: {valid_json_count}/10")

Sentiment v1 (10 runs) — unique outputs: 10/10
  [Run 01] Positive
  [Run 02] Label: Positive sentiment (praise/compliment).
  [Run 03] Label: Positive (strong positive sentiment)
  [Run 04] Sentiment: Positive — strongly positive.
  [Run 05] Positive — very satisfied / enthusiastic (strong positive sentiment).
  [Run 06] Positive — enthusiastic praise / positive feedback. The message expresses strong satisfaction with the product.
  [Run 07] Positive — strongly positive sentiment (customer expresses clear satisfaction and delight).
  [Run 08] Sentiment: Positive
  [Run 09] Sentiment: Positive (very positive).
  [Run 10] Label: Positive (strongly positive)

Product v1 (10 runs) — unique outputs: 10/10
  [Run 01] (1261 chars) Compact comfort. Confident control. Just $29.99.
  [Run 02] (1733 chars) GlidePro Wireless Mouse — $29.99
  [Run 03] (1396 chars) SmoothClick Pro Wireless Mouse — Precision, Comfort, and Value for $29.99
  [Run 04] (1486 chars) Sleek. Reliable. Comfortable. All for

In [9]:
# Define v2 sentiment prompt with strict allowed labels and zero extra commentary
sentiment_prompt_v2 = """
Classify the sentiment of the customer message below.
Respond with ONLY one exact word from this list: positive, negative, neutral.
Do not include punctuation, labels, or explanations.

Customer message: "I love this product! It's exactly what I needed."
"""

# Run a single v2 sentiment check to verify strict single-word output
print("Sentiment v2 single run:", repr(call_openai(sentiment_prompt_v2, sleep_seconds=0.1)))

# Define v2 product prompt with length cap, price inclusion, and anti-hallucination constraints
product_prompt_v2 = """
Write a concise product description (80 to 120 words) for a wireless mouse priced at $29.99.
Include the exact price $29.99.
Do not invent a brand name, model name, DPI numbers, battery life hours, or unmentioned specs.
Use a single short paragraph.

Product: wireless mouse ($29.99)
"""

# Run a single v2 product description check to inspect length and grounding
print("\nProduct v2 single run:\n" + call_openai(product_prompt_v2, sleep_seconds=0.1))

# Define v2 extraction prompt with a fixed JSON schema and raw-JSON-only instruction
extraction_prompt_v2 = """
Extract structured data from the customer feedback below.
Return ONLY valid JSON (no markdown code fences, no intro or outro text) using these exact keys:
- "order_id": string or null
- "order_date": string or null
- "delivery_status": string or null
- "packaging_status": string or null
- "sentiment": string ("positive", "negative", "neutral", or "mixed")

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

# Run a single v2 extraction check to verify raw JSON and exact keys
print("\nExtraction v2 single run:\n" + call_openai(extraction_prompt_v2, sleep_seconds=0.1))

Sentiment v2 single run: 'positive'

Product v2 single run:
This wireless mouse offers comfortable, reliable control at an attractive price of $29.99. Its contoured shape fits everyday use, delivering smooth cursor movement and responsive clicks for browsing and productivity tasks. Designed for simplicity, it requires minimal setup and provides dependable wireless performance without unnecessary complexity. The clean, compact form suits home, office, or travel, while solid construction handles daily use. An affordable, no-nonsense accessory that emphasizes comfort and ease of use, it’s a practical choice for anyone seeking a straightforward wireless pointing solution.

Extraction v2 single run:
{
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_status": "fast",
  "packaging_status": "damaged",
  "sentiment": "mixed"
}


In [10]:
# Run sentiment_prompt_v2 10 times with minimal delay to check label stability
sentiment_v2_10runs = run_multiple_fast(sentiment_prompt_v2, runs=10, sleep_seconds=0.1)
# Print how many unique outputs occurred out of 10 runs
print(f"Sentiment v2 (10 runs) — unique outputs: {len(set(sentiment_v2_10runs))}/10")
# Count how many runs returned the exact expected lowercase label 'positive'
exact_positive_count = sum(1 for out in sentiment_v2_10runs if out.strip() == "positive")
# Print exact-match rate so any punctuation or casing slip is caught immediately
print(f"Exact 'positive' matches: {exact_positive_count}/10")
# Print all 10 raw outputs for inspection
print("All runs:", sentiment_v2_10runs)

Sentiment v2 (10 runs) — unique outputs: 1/10
Exact 'positive' matches: 10/10
All runs: ['positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive']


In [11]:
# Run sentiment_prompt_v2 10 times with minimal delay to check label stability
sentiment_v2_10runs = run_multiple_fast(sentiment_prompt_v2, runs=10, sleep_seconds=0.1)
# Print how many unique outputs occurred out of 10 runs
print(f"Sentiment v2 (10 runs) — unique outputs: {len(set(sentiment_v2_10runs))}/10")
# Count how many runs returned the exact expected lowercase label 'positive'
exact_positive_count = sum(1 for out in sentiment_v2_10runs if out.strip() == "positive")
# Print exact-match rate so any punctuation or casing slip is caught immediately
print(f"Exact 'positive' matches: {exact_positive_count}/10")
# Print all 10 raw outputs for inspection
print("All runs:", sentiment_v2_10runs)

Sentiment v2 (10 runs) — unique outputs: 1/10
Exact 'positive' matches: 10/10
All runs: ['positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive']


In [12]:
# Run product_prompt_v2 10 times to measure word/character consistency and constraint adherence
product_v2_10runs = run_multiple_fast(product_prompt_v2, runs=10, sleep_seconds=0.1)
# Print unique output count out of 10 runs
print(f"Product v2 (10 runs) — unique outputs: {len(set(product_v2_10runs))}/10")
# Inspect each run's word count, $29.99 presence, and opening words
for idx, output in enumerate(product_v2_10runs, start=1):
    # Split text on whitespace to check approximate word count against the 80-120 word target
    word_count = len(output.split())
    # Check whether the exact price $29.99 appears in the output
    has_price = "$29.99" in output
    # Print a compact summary line for each run
    print(f"  [Run {idx:02d}] words={word_count} | has_$29.99={has_price} | start={output[:65]!r}")

Product v2 (10 runs) — unique outputs: 10/10
  [Run 01] words=85 | has_$29.99=True | start='This wireless mouse, priced at $29.99, provides cable-free contro'
  [Run 02] words=95 | has_$29.99=True | start='At $29.99, this wireless mouse delivers comfortable, reliable con'
  [Run 03] words=91 | has_$29.99=True | start='Experience comfortable, precise control with this wireless mouse,'
  [Run 04] words=94 | has_$29.99=True | start='This wireless mouse, priced at $29.99, combines simple, comfortab'
  [Run 05] words=94 | has_$29.99=True | start='This wireless mouse delivers comfortable, responsive control in a'
  [Run 06] words=87 | has_$29.99=True | start='Stylish, comfortable wireless mouse offering smooth, responsive t'
  [Run 07] words=79 | has_$29.99=True | start='This wireless mouse offers comfortable, ergonomic contours and sm'
  [Run 08] words=90 | has_$29.99=True | start='This wireless mouse offers comfortable control and reliable perfo'
  [Run 09] words=91 | has_$29.99=True | sta

In [13]:
# Import json to validate JSON parsing and check schema keys across all 10 runs
import json

# Run extraction_prompt_v2 10 times to test JSON validity and exact key consistency
extraction_v2_10runs = run_multiple_fast(extraction_prompt_v2, runs=10, sleep_seconds=0.1)
# Print unique raw output count out of 10 runs
print(f"Extraction v2 (10 runs) — unique outputs: {len(set(extraction_v2_10runs))}/10")

# Define the exact schema keys expected in every valid extraction result
expected_keys = ["order_id", "order_date", "delivery_status", "packaging_status", "sentiment"]
# Track valid JSON count and exact key-set match count
valid_json_v2 = 0
exact_schema_v2 = 0

# Evaluate every run for parseability and exact schema conformance
for idx, output in enumerate(extraction_v2_10runs, start=1):
    try:
        # Parse the output string as JSON
        parsed = json.loads(output)
        # Increment valid JSON counter
        valid_json_v2 += 1
        # Check whether the parsed object's keys match expected_keys exactly
        keys_match = isinstance(parsed, dict) and sorted(parsed.keys()) == sorted(expected_keys)
        # Increment exact schema counter when all keys match
        if keys_match:
            exact_schema_v2 += 1
        # Print status and keys for this run
        print(f"  [Run {idx:02d}] Valid JSON | exact_keys={keys_match} | keys={list(parsed.keys())}")
    except json.JSONDecodeError:
        # Print diagnostic info if a run returned non-JSON text or code fences
        print(f"  [Run {idx:02d}] INVALID JSON | output={output[:80]!r}")

# Print summary metrics to compare directly against v1's 5/10 valid JSON rate
print(f"Extraction v2 valid JSON rate: {valid_json_v2}/10 | exact schema rate: {exact_schema_v2}/10")

Extraction v2 (10 runs) — unique outputs: 4/10
  [Run 01] Valid JSON | exact_keys=True | keys=['order_id', 'order_date', 'delivery_status', 'packaging_status', 'sentiment']
  [Run 02] Valid JSON | exact_keys=True | keys=['order_id', 'order_date', 'delivery_status', 'packaging_status', 'sentiment']
  [Run 03] Valid JSON | exact_keys=True | keys=['order_id', 'order_date', 'delivery_status', 'packaging_status', 'sentiment']
  [Run 04] Valid JSON | exact_keys=True | keys=['order_id', 'order_date', 'delivery_status', 'packaging_status', 'sentiment']
  [Run 05] Valid JSON | exact_keys=True | keys=['order_id', 'order_date', 'delivery_status', 'packaging_status', 'sentiment']
  [Run 06] Valid JSON | exact_keys=True | keys=['order_id', 'order_date', 'delivery_status', 'packaging_status', 'sentiment']
  [Run 07] Valid JSON | exact_keys=True | keys=['order_id', 'order_date', 'delivery_status', 'packaging_status', 'sentiment']
  [Run 08] Valid JSON | exact_keys=True | keys=['order_id', 'order_date

In [14]:
# Define v3 sentiment prompt with few-shot examples to reinforce strict single-label output
sentiment_prompt_v3 = """
Classify the customer message sentiment.
Output ONLY one exact word: positive, negative, or neutral. No punctuation, no extra text.

Examples:
Message: "My order arrived two days late and the box was crushed." -> negative
Message: "I received my package today." -> neutral
Message: "I love this product! It's exactly what I needed." -> positive

Message: "I love this product! It's exactly what I needed."
"""

# Run sentiment_prompt_v3 15 times to stress-test label consistency at larger sample size
sentiment_v3_15runs = run_multiple_fast(sentiment_prompt_v3, runs=15, sleep_seconds=0.1)
# Print unique output count out of 15 runs
print(f"Sentiment v3 (15 runs) — unique outputs: {len(set(sentiment_v3_15runs))}/15")
# Count exact 'positive' responses across all 15 runs
exact_pos_v3 = sum(1 for r in sentiment_v3_15runs if r.strip() == "positive")
# Display exact match count and raw list
print(f"Exact 'positive' matches: {exact_pos_v3}/15")
print("All runs:", sentiment_v3_15runs)

Sentiment v3 (15 runs) — unique outputs: 1/15
Exact 'positive' matches: 15/15
All runs: ['positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive']


In [15]:
# Define v3 product description prompt with a slightly safer word range and strict grounding
product_prompt_v3 = """
Write a single-paragraph product description for a wireless mouse priced at $29.99.
Target length: 85 to 115 words.
Rules:
- Include the exact price $29.99.
- Do not invent a brand name, model name, DPI numbers, battery hours, or unstated specs.
- Keep the tone helpful, clear, and grounded in the prompt.

Input: wireless mouse, price $29.99
"""

# Run product_prompt_v3 15 times to check length and price adherence across 15 runs
product_v3_15runs = run_multiple_fast(product_prompt_v3, runs=15, sleep_seconds=0.1)
# Print unique output count out of 15 runs
print(f"Product v3 (15 runs) — unique outputs: {len(set(product_v3_15runs))}/15")
# Track how many runs satisfy the 85-115 word window and include $29.99
in_range_count = 0
price_ok_count = 0

# Inspect each run individually
for idx, output in enumerate(product_v3_15runs, start=1):
    # Count words via whitespace splitting
    wc = len(output.split())
    # Check whether exact price string is present
    has_price = "$29.99" in output
    # Check whether word count falls inside 85..115 inclusive
    in_range = 85 <= wc <= 115
    # Update counters for summary reporting
    if in_range:
        in_range_count += 1
    if has_price:
        price_ok_count += 1
    # Print per-run summary
    print(f"  [Run {idx:02d}] words={wc} (in_85_115={in_range}) | has_$29.99={has_price}")

# Print overall compliance counts out of 15
print(f"Word-count compliance (85-115): {in_range_count}/15 | Price ($29.99) compliance: {price_ok_count}/15")

Product v3 (15 runs) — unique outputs: 15/15
  [Run 01] words=95 (in_85_115=True) | has_$29.99=True
  [Run 02] words=93 (in_85_115=True) | has_$29.99=True
  [Run 03] words=95 (in_85_115=True) | has_$29.99=True
  [Run 04] words=90 (in_85_115=True) | has_$29.99=True
  [Run 05] words=93 (in_85_115=True) | has_$29.99=True
  [Run 06] words=92 (in_85_115=True) | has_$29.99=True
  [Run 07] words=95 (in_85_115=True) | has_$29.99=True
  [Run 08] words=94 (in_85_115=True) | has_$29.99=True
  [Run 09] words=93 (in_85_115=True) | has_$29.99=True
  [Run 10] words=92 (in_85_115=True) | has_$29.99=True
  [Run 11] words=92 (in_85_115=True) | has_$29.99=True
  [Run 12] words=92 (in_85_115=True) | has_$29.99=True
  [Run 13] words=96 (in_85_115=True) | has_$29.99=True
  [Run 14] words=96 (in_85_115=True) | has_$29.99=True
  [Run 15] words=100 (in_85_115=True) | has_$29.99=True
Word-count compliance (85-115): 15/15 | Price ($29.99) compliance: 15/15


In [16]:
# Import json to verify JSON parsing and schema accuracy across 15 runs
import json

# Define v3 extraction prompt with explicit example and strict raw-JSON-only constraint
extraction_prompt_v3 = """
Extract structured fields from the customer feedback below.
Return ONLY a raw JSON object (no markdown fences, no prose before or after) with these exact keys:
"order_id", "order_date", "delivery_status", "packaging_status", "sentiment".

Example feedback: "Placed order #9876 on April 2. Delivery was slow and box arrived torn."
Example output:
{"order_id": "9876", "order_date": "April 2", "delivery_status": "slow", "packaging_status": "torn", "sentiment": "negative"}

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

# Run extraction_prompt_v3 15 times to stress-test JSON parsing and schema stability
extraction_v3_15runs = run_multiple_fast(extraction_prompt_v3, runs=15, sleep_seconds=0.1)
# Print unique raw output count out of 15 runs
print(f"Extraction v3 (15 runs) — unique outputs: {len(set(extraction_v3_15runs))}/15")

# Define the required schema keys
expected_keys_v3 = ["order_id", "order_date", "delivery_status", "packaging_status", "sentiment"]
# Track valid JSON parses and exact schema matches
valid_json_v3 = 0
exact_schema_v3 = 0

# Check every single one of the 15 outputs
for idx, output in enumerate(extraction_v3_15runs, start=1):
    try:
        # Parse JSON output
        parsed = json.loads(output)
        # Count valid JSON parse
        valid_json_v3 += 1
        # Check whether keys match expected_keys_v3 exactly
        exact_match = isinstance(parsed, dict) and sorted(parsed.keys()) == sorted(expected_keys_v3)
        # Count exact schema matches
        if exact_match:
            exact_schema_v3 += 1
        # Print status and parsed object
        print(f"  [Run {idx:02d}] Valid JSON | exact_keys={exact_match} | data={parsed}")
    except json.JSONDecodeError:
        # Log any run that failed JSON parsing
        print(f"  [Run {idx:02d}] INVALID JSON | raw={output[:80]!r}")

# Print 15-run summary metrics
print(f"Extraction v3 valid JSON: {valid_json_v3}/15 | exact schema: {exact_schema_v3}/15")

Extraction v3 (15 runs) — unique outputs: 5/15
  [Run 01] Valid JSON | exact_keys=True | data={'order_id': '12345', 'order_date': 'March 15th', 'delivery_status': 'fast', 'packaging_status': 'damaged', 'sentiment': 'mixed'}
  [Run 02] Valid JSON | exact_keys=True | data={'order_id': '12345', 'order_date': 'March 15th', 'delivery_status': 'fast', 'packaging_status': 'damaged', 'sentiment': 'negative'}
  [Run 03] Valid JSON | exact_keys=True | data={'order_id': '12345', 'order_date': 'March 15th', 'delivery_status': 'fast', 'packaging_status': 'damaged', 'sentiment': 'mixed'}
  [Run 04] Valid JSON | exact_keys=True | data={'order_id': '12345', 'order_date': 'March 15th', 'delivery_status': 'fast', 'packaging_status': 'damaged', 'sentiment': 'mixed'}
  [Run 05] Valid JSON | exact_keys=True | data={'order_id': '12345', 'order_date': 'March 15th', 'delivery_status': 'fast', 'packaging_status': 'damaged', 'sentiment': 'mixed'}
  [Run 06] Valid JSON | exact_keys=True | data={'order_id': '1234

In [1]:
import json
import re


def evaluate_batch(
    outputs,
    task,
    required_keys=None,
    word_range=(85, 115),
    required_price="$29.99",
    allowed_sentiments=("positive", "negative", "neutral", "mixed"),
):
    # Track overall run count and how many distinct strings came back across the batch
    total = len(outputs)
    unique_outputs = set(outputs)

    # Counters for format validity (structure/length/JSON) and content validity (fields/price/label)
    format_passes = 0
    content_passes = 0
    details = []

    for idx, raw in enumerate(outputs, start=1):
        # Normalize whitespace so accidental leading/trailing spaces don't break checks
        text = raw.strip() if isinstance(raw, str) else ""
        fmt_ok = False
        cnt_ok = False

        if task == "data_extraction":
            try:
                # Check that the output parses cleanly as a JSON object (dict)
                parsed = json.loads(text)
                fmt_ok = isinstance(parsed, dict)

                # Default to the lab's 5 extraction fields unless custom keys are passed in
                expected = set(
                    required_keys
                    or [
                        "order_id",
                        "order_date",
                        "delivery_status",
                        "packaging_status",
                        "sentiment",
                    ]
                )
                # Verify exact key match (no missing or unexpected keys)
                cnt_ok = fmt_ok and (set(parsed.keys()) == expected)
            except Exception:
                # Invalid JSON fails both format and content checks
                fmt_ok = False
                cnt_ok = False

        elif task == "product_description":
            # Count word tokens and ensure output stays as a single block (no blank-line paragraph split)
            words = re.findall(r"\b\w+\b", text)
            lo, hi = word_range
            fmt_ok = (lo <= len(words) <= hi) and ("\n\n" not in text)

            # Check that the required price string appears verbatim in the description
            cnt_ok = (required_price in text) if required_price else True

        elif task == "sentiment":
            # Strip common outer punctuation/casing to check if a single clean label was returned
            cleaned = text.lower().strip(" .!\"'")
            fmt_ok = (len(cleaned.split()) == 1) and (
                cleaned == text.lower().strip(" .!\"'")
            )
            # Confirm the label belongs to the allowed sentiment set
            cnt_ok = cleaned in {s.lower() for s in allowed_sentiments}

        format_passes += int(fmt_ok)
        content_passes += int(cnt_ok)
        details.append(
            {
                "run": idx,
                "format_ok": fmt_ok,
                "content_ok": cnt_ok,
                "preview": text[:90],
            }
        )

    return {
        "task": task,
        "total_runs": total,
        "unique_outputs": len(unique_outputs),
        # Higher means fewer distinct outputs across repeats (0.0 = every run differed, 1.0 = identical)
        "consistency_ratio": round(1 - (len(unique_outputs) / total), 3) if total else 0.0,
        "format_pass_rate": round(format_passes / total, 3) if total else 0.0,
        "content_pass_rate": round(content_passes / total, 3) if total else 0.0,
        "details": details,
    }

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from a .env file in the current or parent directory if present
load_dotenv()

# Create the OpenAI client using OPENAI_API_KEY from the environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or .env file.")

client = OpenAI(api_key=api_key)
MODEL_NAME = globals().get("MODEL", "gpt-5-mini")
print("Client initialized cleanly using model:", MODEL_NAME)

Client initialized cleanly using model: gpt-5-mini


In [5]:
# Reuse existing client/model from earlier cells if defined, otherwise set up a fallback
try:
    client
except NameError:
    from openai import OpenAI
    client = OpenAI()

MODEL_NAME = globals().get("MODEL", "gpt-5-mini")
SAMPLE_EXTRACTION_INPUT = (
    "I ordered item #12345 on March 15th. "
    "The delivery was fast but the packaging was damaged."
)
EXTRACTION_PROMPT = (
    "Extract order_id, order_date, delivery_status, packaging_status, and sentiment "
    "as valid JSON only. Input: " + SAMPLE_EXTRACTION_INPUT
)


def single_call(prompt, temperature=None, reasoning_effort=None):
    # Build request kwargs dynamically so optional parameters are only sent when requested
    kwargs = {
        "model": MODEL_NAME,
        "input": prompt,
    }
    if temperature is not None:
        kwargs["temperature"] = temperature
    if reasoning_effort is not None:
        kwargs["reasoning"] = {"effort": reasoning_effort}

    # Support both Responses API and Chat Completions API depending on SDK version
    if hasattr(client, "responses"):
        resp = client.responses.create(**kwargs)
        return resp.output_text
    else:
        chat_kwargs = {"model": MODEL_NAME, "messages": [{"role": "user", "content": prompt}]}
        if temperature is not None:
            chat_kwargs["temperature"] = temperature
        resp = client.chat.completions.create(**chat_kwargs)
        return resp.choices[0].message.content


# Step A: Single probe call at temperature=0 to check if custom temperature is supported
temperature_supported = False
try:
    _probe = single_call(EXTRACTION_PROMPT, temperature=0)
    temperature_supported = True
    print("Temperature probe succeeded: custom temperature values are supported.")
except Exception as err:
    # Expected on gpt-5-mini where temperature is fixed to 1
    print("Temperature probe rejected (expected on gpt-5-mini):", err)

results = {}

if temperature_supported:
    # Sweep temperatures 0, 0.3, 0.7, and 1.0 with 3 runs each to keep API calls low
    for temp in [0, 0.3, 0.7, 1.0]:
        batch_outputs = [single_call(EXTRACTION_PROMPT, temperature=temp) for _ in range(3)]
        results[f"temp={temp}"] = evaluate_batch(batch_outputs, task="data_extraction")
else:
    # Since temperature is locked to 1 on gpt-5-mini, test reasoning_effort with 3 runs each
    for effort in ["minimal", "low", "medium"]:
        try:
            batch_outputs = [
                single_call(EXTRACTION_PROMPT, reasoning_effort=effort) for _ in range(3)
            ]
            results[f"reasoning_effort={effort}"] = evaluate_batch(
                batch_outputs, task="data_extraction"
            )
        except Exception as err:
            # Record if a specific reasoning_effort level is unsupported by the model
            results[f"reasoning_effort={effort}"] = {"error": str(err)}

# Print evaluation summary for each tested setting
for setting, summary in results.items():
    print(f"\n--- {setting} ---")
    print(summary)

Temperature probe rejected (expected on gpt-5-mini): Error code: 400 - Unsupported parameter: 'temperature' is not supported with this model.

--- reasoning_effort=minimal ---
{'task': 'data_extraction', 'total_runs': 3, 'unique_outputs': 3, 'consistency_ratio': 0.0, 'format_pass_rate': 1.0, 'content_pass_rate': 1.0, 'details': [{'run': 1, 'format_ok': True, 'content_ok': True, 'preview': '{\n  "order_id": "12345",\n  "order_date": "March 15",\n  "delivery_status": "fast",\n  "packa'}, {'run': 2, 'format_ok': True, 'content_ok': True, 'preview': '{"order_id":"12345","order_date":"March 15","delivery_status":"fast","packaging_status":"d'}, {'run': 3, 'format_ok': True, 'content_ok': True, 'preview': '{\n  "order_id": "12345",\n  "order_date": "2023-03-15",\n  "delivery_status": "fast",\n  "pac'}]}

--- reasoning_effort=low ---
{'task': 'data_extraction', 'total_runs': 3, 'unique_outputs': 2, 'consistency_ratio': 0.333, 'format_pass_rate': 1.0, 'content_pass_rate': 1.0, 'details': [{'run

In [6]:
# Baseline v1-style extraction prompt and known issues from the earlier code/output audit
V1_EXTRACTION_PROMPT = (
    "Extract the order details from the customer text and return JSON with "
    "order_id, order_date, delivery_status, packaging_status, and sentiment."
)

FAILURE_NOTES = [
    "May include markdown fences or extra commentary outside raw JSON.",
    "Inconsistent date formatting or inventing a year not present in the source text.",
    "No explicit rule for handling ambiguous/mixed sentiment or keeping exact field values.",
]

# Meta-prompt asking the model to rewrite V1_EXTRACTION_PROMPT while addressing each failure point
META_PROMPT = (
    "You are a prompt engineer. Improve the baseline extraction prompt below so it produces "
    "strictly valid, raw JSON (no markdown code fences, no extra text) with exact keys: "
    "order_id, order_date, delivery_status, packaging_status, sentiment.\n"
    "Address these failure modes:\n- " + "\n- ".join(FAILURE_NOTES) + "\n\n"
    "Baseline prompt:\n" + V1_EXTRACTION_PROMPT + "\n\n"
    "Return ONLY the improved prompt text."
)

# Step 1: Generate the meta-improved prompt (1 API call)
meta_resp = client.responses.create(
    model=MODEL_NAME,
    input=META_PROMPT,
    reasoning={"effort": "low"},
)
v_meta_prompt = meta_resp.output_text.strip()
print("=== Generated v_meta Prompt ===\n")
print(v_meta_prompt)

# Step 2: Run v_meta on the sample input across 3 runs and evaluate with evaluate_batch
meta_test_input = f"{v_meta_prompt}\n\nCustomer text:\n{SAMPLE_EXTRACTION_INPUT}"
meta_outputs = [
    client.responses.create(
        model=MODEL_NAME,
        input=meta_test_input,
        reasoning={"effort": "low"},
    ).output_text
    for _ in range(3)
]

meta_eval = evaluate_batch(meta_outputs, task="data_extraction")
print("\n=== v_meta Evaluation ===")
print(meta_eval)

=== Generated v_meta Prompt ===

You are a JSON extractor. Read the customer text and output strictly valid raw JSON only (no surrounding text, no markdown fences, no explanations). The JSON object must contain exactly these five keys in this order: order_id, order_date, delivery_status, packaging_status, sentiment.

Rules for each key:
- order_id: extract the exact substring from the text that represents the order identifier. Do not normalize, add, or invent characters. If no order id is present, set value to null.
- order_date: extract the exact date substring as it appears in the text. Do not normalize formatting or invent a year. If the text contains multiple date-like substrings and it is ambiguous which is the order date, set value to null. If no date-like substring is present, set value to null.
- delivery_status: map the message to one of these exact strings: "delivered", "in_transit", "not_delivered", "unknown". Determine intent from the text; if the message clearly states the

In [7]:
# Sample inputs for each of the three lab tasks
TASK_INPUTS = {
    "sentiment": (
        "The delivery was super fast and the support team was helpful, "
        "though the outer box arrived slightly dented."
    ),
    "product_description": (
        "Product: PocketBrew Travel Mug | Price: $29.99 | "
        "Features: double-wall vacuum insulation, leak-proof lid, fits cup holders. "
        "Write a single paragraph of about 95-105 words without inventing specs."
    ),
    "data_extraction": SAMPLE_EXTRACTION_INPUT,
}

# Single unified prompt that routes behavior based on the requested `task` name
MULTI_TASK_PROMPT_TEMPLATE = """You are a multi-task assistant. Follow ONLY the instructions for the specified task.
Global rules:
- Never invent facts, specifications, dates, or prices not supported by the input.
- Do not wrap output in markdown code fences unless explicitly asked.

Task: {task}

If task == "sentiment":
- Output ONLY a single lowercase word from: positive, negative, neutral, mixed.
- Use "mixed" when both clearly positive and clearly negative aspects are present.

If task == "product_description":
- Write a single paragraph (no line breaks or bullet points) of roughly 90-110 words.
- Include the exact price $29.99 and use only features/facts from the input.

If task == "data_extraction":
- Output strictly valid raw JSON with exact keys:
  order_id, order_date, delivery_status, packaging_status, sentiment.
- Do not invent missing dates/years; use null or "unknown" when unspecified.

Input:
{input_text}
"""


def run_multi_task(task_name, input_text, runs=2):
    # Format the shared multi-task prompt for one specific task and input
    prompt = MULTI_TASK_PROMPT_TEMPLATE.format(task=task_name, input_text=input_text)
    return [
        client.responses.create(
            model=MODEL_NAME,
            input=prompt,
            reasoning={"effort": "low"},
        ).output_text
        for _ in range(runs)
    ]


multi_results = {}

# Run 2 repeats per task (6 calls total) and score each batch with evaluate_batch
sentiment_outs = run_multi_task("sentiment", TASK_INPUTS["sentiment"], runs=2)
multi_results["sentiment"] = evaluate_batch(sentiment_outs, task="sentiment")

desc_outs = run_multi_task("product_description", TASK_INPUTS["product_description"], runs=2)
multi_results["product_description"] = evaluate_batch(
    desc_outs, task="product_description", word_range=(85, 115), required_price="$29.99"
)

extract_outs = run_multi_task("data_extraction", TASK_INPUTS["data_extraction"], runs=2)
multi_results["data_extraction"] = evaluate_batch(extract_outs, task="data_extraction")

for task_name, summary in multi_results.items():
    print(f"\n=== Multi-Task: {task_name} ===")
    print(summary)


=== Multi-Task: sentiment ===
{'task': 'sentiment', 'total_runs': 2, 'unique_outputs': 1, 'consistency_ratio': 0.5, 'format_pass_rate': 1.0, 'content_pass_rate': 1.0, 'details': [{'run': 1, 'format_ok': True, 'content_ok': True, 'preview': 'mixed'}, {'run': 2, 'format_ok': True, 'content_ok': True, 'preview': 'mixed'}]}

=== Multi-Task: product_description ===
{'task': 'product_description', 'total_runs': 2, 'unique_outputs': 2, 'consistency_ratio': 0.0, 'format_pass_rate': 1.0, 'content_pass_rate': 1.0, 'details': [{'run': 1, 'format_ok': True, 'content_ok': True, 'preview': 'PocketBrew Travel Mug is a compact, everyday companion priced at $29.99 that combines prac'}, {'run': 2, 'format_ok': True, 'content_ok': True, 'preview': 'The PocketBrew Travel Mug is designed for everyday convenience and dependable performance,'}]}

=== Multi-Task: data_extraction ===
{'task': 'data_extraction', 'total_runs': 2, 'unique_outputs': 2, 'consistency_ratio': 0.0, 'format_pass_rate': 1.0, 'content_p